In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# 冻结 LAST 基线：存在性、片段及序列判定

Run all固定9个新calibration OFF源，锁定source-max阈值后，再执行2新内容×OFF/A/B的6个evaluation源。
模型/写入/接收器不改，只使用完整blind local_state搜索后的score作存在统计。
存在接受、消息唯一/正确、同步窗口对应分开；A/B排名不等于检测。
每源预定7view：full181、crop0/4/5各129、delete90单帧、speed1.25近邻145帧、secondH264。
三攻击均第二次CRF18 H264；crop无第二次codec。所有view均从落盘真实RGB读回再VAE编码4phase。
序列统计固定对同message的3crop最佳clock分数取平均，再选message最大值，不选最佳片段。
三个重叠crop为同一源内证据，不是3独立样本。

每cal源取7view及sequence最大值；9源最大值为tau，严格score>tau。任一calview缺失则UNCALIBRATED，
仍保存eval排名但存在判定UNDECIDED。阈值及SHA在任何eval模型加载前固定，不据结果改参数。
9手选场景并非总体随机样本；.1只是条件交换性假设下的最粗rank分辨率，不是低FPR保证。
OFF按攻击view及source-any误接受另报；缺失不当阴性。受攻击位置不进入blind score。

固定15source/105view/420encode、1100Transformer/554native、15decode/60MP4save/45cropnpy。
新输出FlowTubeDetection；源ZIP、配置、阈值、完整失败槽与结果落盘。
质量PSNR/时间残差只衡量与同场景OFF差异；eval6个完整base+18受攻击完整视频匿名拷贝至blind_review，
ratings.csv的成像质量/主体完整性/运动平滑三项均PENDING。先盲评，勿提前看reporting_only_review_mapping.json。
任何统计/CPU测试都不自动填写人工质量PASS。
源码SHA：f561fd58f01c8cc44a05885c0eb9ff42017465c7。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.metadata, json, os, signal, subprocess, sys
SOURCE_COMMIT = 'f561fd58f01c8cc44a05885c0eb9ff42017465c7'
if SOURCE_COMMIT is None:
    raise RuntimeError('Pending source publication: rebuild with the published full SHA')
SOURCE_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
RUN_ID = 'flow_tube_detection_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SOURCE = Path('/content') / (RUN_ID + '_source')
subprocess.run(['git', 'init', str(SOURCE)], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'remote', 'add', 'origin', SOURCE_URL], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', '--depth', '1', 'origin', SOURCE_COMMIT], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', SOURCE_COMMIT], check=True)
if subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip() != SOURCE_COMMIT:
    raise RuntimeError('Source checkout differs from pinned SHA')
def version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None
if version('torch') != '2.11.0+cu128':
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.11.0', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers==0.40.0', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
subprocess.run([sys.executable, '-c', "import torch,diffusers; assert str(torch.__version__) == '2.11.0+cu128', torch.__version__; assert diffusers.__version__ == '0.40.0', diffusers.__version__"], check=True)


In [ ]:
OUTPUT = Path('/content/drive/MyDrive/Video-WM/FlowTubeDetection') / RUN_ID
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
ARCHIVE = OUTPUT.parent / (RUN_ID + '.source.zip')
subprocess.run(['git', '-C', str(SOURCE), 'archive', '--format=zip', '--output', str(ARCHIVE), SOURCE_COMMIT], check=True)
LOG = OUTPUT.parent / (RUN_ID + '.launcher.log')
command = [sys.executable, '-u', '-m', 'experiments.wan_state_clock.flow_tube_detection_run', '--output', str(OUTPUT)]
with LOG.open('w') as log:
    process = subprocess.Popen(command, cwd=SOURCE, start_new_session=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
        code = process.wait()
    except BaseException:
        try:
            os.killpg(process.pid, signal.SIGTERM); process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL); process.wait()
        except ProcessLookupError:
            pass
        raise
print('Output:', OUTPUT, 'Launcher log:', LOG, 'Source archive:', ARCHIVE)
if (OUTPUT / 'result.json').exists():
    result = json.loads((OUTPUT / 'result.json').read_text())
    print(json.dumps({k:result.get(k) for k in ('status','source_denominator','view_denominator','receiver_encode_denominator','fixed_calls','actual_calls','threshold','summary','human_review')}, indent=2))
if code:
    raise subprocess.CalledProcessError(code, command)
